# W8-1. Feature Selection & SVM

**오늘 할 일**
1. 저번 주 Logistic Regression 흐름을 다시 한 번 따라가 보기
2. `SelectKBest`로 descriptor(=feature) **몇 개**를 쓸지 정하기
3. 같은 개수로 **MLP**, **SVM**도 학습시켜서 성능 비교

> 📝 **descriptor = feature** — 분자 구조를 숫자로 바꾼 값. 동일한 의미.

**데이터**: `skin_irritation_2Ddesc.csv` (저번 시간에 만든 파일)

**구성**
- Part A — 저번 주 코드 복습 (셀 그대로 실행)
- Part B — 이번 주 실습 (✏️ 표시 셀은 직접 채워 넣기)


---
# Part A. 저번 주 복습

셀을 하나씩 `Shift+Enter`로 실행하면서 주석 읽어 봐.


## A-1. 데이터 불러오기

`pandas.read_csv`로 파일 읽고 `df`에 담기.
- `df.shape` → (행, 열)
- `df.head()` → 위 5줄 미리보기

✅ **예상**: `(39, 220)` — 화합물 39개, 열 220개.


In [ ]:
import pandas as pd

df = pd.read_csv('skin_irritation_2Ddesc.csv')
print('shape:', df.shape)
df.head()


## A-2. X / y 나누고 descriptor 정리

- `y` = `label` 열 (피부 자극: 0/1)
- `X` = 나머지 숫자 descriptor

버릴 거 두 가지:
1. **NaN 있는 열** — 일부 descriptor는 특정 분자에서 계산이 실패해 `NaN`이 돼. 👉 저번 주 마지막에 `clf.fit(X, y)`가 터진 이유가 바로 이거야.
2. **표준편차 0.01 미만 열** — 거의 다 같은 값이면 구분력이 없으니 빼.

✅ **예상**: NaN 제거 `(39, 209)` → std 필터 `(39, 144)`


In [ ]:
y = df['label']
X = df.drop(columns=['Chemical_Name', 'standardized_smi', 'label'])

X = X.dropna(axis=1)                 # NaN 열 제거
print('NaN 제거:', X.shape)

X = X.loc[:, X.std() >= 0.01]        # 저분산 제거
print('std 필터:', X.shape)


## A-3. 전체 descriptor로 Logistic Regression

일단 다 넣고 돌려 봐.
- `clf.fit(X, y)` : 학습
- `clf.score(X, y)` : 학습 데이터 정확도 — 1.0에 가까워도 좋아하면 안 돼(**overfitting** 의심).
- `cross_val_score(..., cv=5)` : 5등분해서 돌아가며 검증. 진짜 성능은 이쪽.
- 함수 설명: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html
- cross validation 설명: https://scikit-learn.org/stable/modules/cross_validation.html

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

'''
descriptor 단위가 달라서 표준화 (사실 2D descriptor 사용할 때는 normalization이 필수!)
https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html

standardscaler는 descriptor 값을 평균과 표준편차로 표준화시키는 방법. (나는 고등학교 때 수학 시간에 확률과 통계 시간에 배웠던 것 같은데...)

또 다른 표준화 방식은 descriptor의 최대, 최소 값을 이용하는 것.
https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html
'''

X_scaled = StandardScaler().fit_transform(X)   # descriptor값 표준화

clf = LogisticRegression(max_iter=5000)
clf.fit(X_scaled, y)
print('train :', clf.score(X_scaled, y))
print('CV5   :', cross_val_score(clf, X_scaled, y, cv=5).mean().round(3))



---
# Part B. 이번 주 실습

✏️ 셀 3개를 직접 채워. 바로 위 Part A를 참고하면 충분해.

1. `SelectKBest` + Logistic Regression으로 **최적 K** 찾기
2. 같은 K로 **MLP** — 로지스틱보다 나은가?
3. 같은 K로 **SVM** — `C`, `kernel` 바꿔 보기


In [ ]:
SelectKBest : 데이터 개수(39개)에 비해 특징(144개)이 너무 많으면 모델이 과적합(onerfitting)될 가능성이 크다.
    통계적 방법을 사용하여 정답(label)과 가장 관련이 깊은 상위 k개의 특징만 선택
    K값을 5부터 50까지 바꿔가며, 교차 검증(Cross Validation) 점수가 가장 높은 최적의 특징 개수를 찾는다.
MLP : 인공신경망. 여러 개의 '층'을 쌓아 데이터의 복잡한 패턴을 학습
      데이터 양이 많을수록 유리, hiddden_layer_sizes설정 요
SVM : 데이터들 사이에 가장 넓은 길을 내어 두 그룹으로 나누는 방식

## B-0. SelectKBest 개념

> 📌 **성능 숫자 해석 팁**: 샘플이 39개뿐이니까 5-fold CV는 fold당 8개 정도만 들어가. **±0.1 변동은 정상**이야. 미세한 차이로 모델 좋고 나쁨을 단정하지 마.

데이터는 39개, descriptor는 144개. 입력이 너무 많으면 노이즈에 묻혀서 패턴을 못 잡아. `SelectKBest`는 **y와 관련 높은 상위 K개**만 골라.

- 📖 <https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html>

> ⚠️ `SelectKBest(f_classif, ...)`는 스케일 영향을 안 받아서 원본 `X`에 써도 돼. 대신 **모델 학습 직전엔 반드시 `StandardScaler`**. 순서: `SelectKBest` → `StandardScaler` → 모델.

**사용법** (3줄):
```python
from sklearn.feature_selection import SelectKBest, f_classif
selector = SelectKBest(f_classif, k=10)
X_new = selector.fit_transform(X, y)
```

K=10으로 뽑히는 descriptor 이름 먼저 확인.


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(f_classif, k=10)
selector.fit(X, y)
print(list(X.columns[selector.get_support()]))


## ✏️ 실습 1 — Logistic Regression 최적 K

K를 바꿔 가며 5-fold CV 정확도 비교. 제일 높은 K가 **`best_k`**.

**힌트**
1. `for k in k_candidates:` 루프
2. 루프 안에서 `SelectKBest(f_classif, k)` → `StandardScaler` → `LogisticRegression(max_iter=5000)` 의 CV 점수
3. 가장 높은 k를 `best_k`에 저장 (다음 실습에서 재사용)

✅ **경향**: 너무 적으면 정보 부족, 너무 많으면 노이즈.


In [2]:
import pandas as pd

# 1. 데이터 불러오기 (파일명이 정확한지 확인하세요!)
df = pd.read_csv('skin_irritation_2Ddesc.csv')

# 2. X, y 나누기
y = df['label']
X = df.drop(columns=['Chemical_Name', 'standardized_smi', 'label'])

# 3. 데이터 청소 (NaN 제거 및 저분산 제거)
X = X.dropna(axis=1)                 # 결측치가 있는 열 제거
X = X.loc[:, X.std() >= 0.01]        # 변화가 거의 없는 열 제거

print('데이터 준비 완료!')
print('X shape:', X.shape)
print('y shape:', y.shape)

# --- 이제 아래에 실습 1번 코드를 붙여넣으시면 됩니다 ---

데이터 준비 완료!
X shape: (39, 144)
y shape: (39,)


In [4]:
# TODO: 각 k에 대해 SelectKBest → StandardScaler → LogReg CV5
#       k, acc 출력 + best_k/best_score 업데이트
#       예시 출력: 'k=10: 0.793'

1. SelectKBest(특징고르기) : 144개의 데이터(Descriptor)중에서 정답과 가장 관련이 높은 k개만 선별
2. StandardScaler : 골라낸 k개의 데이터들이 서로 단위가 다르니까 모두 평균0, 표준편차 1로 맞춘다.
3. LogReg : 준비된 데이터를 로지스틱 회귀 모델에 넣어 공부시킴
4. CV5 : 데이터를 5덩어리로 나눠서, 4덩어리로 공부하고 1덩어리로 시험 보는 과정을 5번 반복해 평균 점수를 냄.

In [6]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

k_candidates = [5, 10, 15, 20, 25, 30, 40, 50]
best_k, best_score = None, -1

# TODO: 각 k에 대해 SelectKBest → StandardScaler → LogReg CV5
#       k, acc 출력 + best_k/best_score 업데이트
#       예시 출력: 'k=10: 0.793'

for k in k_candidates:
    # 1. K개 특징 선택
    selector = SelectKBest(f_classif, k=k)           #통계적으로 정답(LABEL)과 관련이 깊은 특징 K개를 자동으로 골라냄
    X_new = selector.fit_transform(X, y)             
    
    # 2. 데이터 표준화 (반드시 학습 직전에 수행)
    X_scaled = StandardScaler().fit_transform(X_new)     #골라낸 테이터들의 수치 범위를 일정하게 맞춘다.
    
    # 3. 모델 정의 및 교차 검증 (CV=5)
    model = LogisticRegression(max_iter=5000)
    scores = cross_val_score(model, X_scaled, y, cv=5)
    avg_score = scores.mean()
    
    print(f'k={k}: {avg_score.round(3)}')
    
    # 최적의 K 업데이트
    if avg_score > best_score:
        best_score = avg_score
        best_k = k

print(f'\n최적의 K: {best_k}, 최고 점수: {best_score.round(3)}')

k=5: 0.668
k=10: 0.793
k=15: 0.793
k=20: 0.793
k=25: 0.743
k=30: 0.743
k=40: 0.664
k=50: 0.661

최적의 K: 10, 최고 점수: 0.793


## ✏️ 실습 2 — 같은 K로 MLP

실습 1의 `best_k` 그대로. 모델만 `LogisticRegression` → `MLPClassifier`.

**힌트**
- `from sklearn.neural_network import MLPClassifier`
- `MLPClassifier(random_state=0, max_iter=2000)`
- hidden layer 1개, hidden node 개수는 k와 동일하게 설정
- 데이터가 적으면 MLP가 로지스틱보다 **떨어질 수도** 있어. 그것도 배움이야.


In [7]:
from sklearn.neural_network import MLPClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# TODO: best_k로 SelectKBest → StandardScaler → MLPClassifier(random_state=0, max_iter=2000)
#       CV5 점수 출력 + 실습 1 결과와 한 줄 비교

from sklearn.neural_network import MLPClassifier

# 1. 최적의 K로 데이터 추출 및 표준화
selector = SelectKBest(f_classif, k=best_k)
X_sel = selector.fit_transform(X, y)
X_scaled = StandardScaler().fit_transform(X_sel)

# 2. MLP 모델 설정 (hidden_layer_sizes=(best_k,) 로 설정)
mlp = MLPClassifier(hidden_layer_sizes=(best_k,), random_state=0, max_iter=2000)    
# 은닉층(hidden layer)의 노드 수를 우리가 찾은 k_best와 똑같이 설정하여 로지스틱 회귀와 공정한 조건에서 비교.

# 3. 성능 확인
mlp_score = cross_val_score(mlp, X_scaled, y, cv=5).mean()
print(f'MLP (k={best_k}) CV5 점수: {mlp_score.round(3)}')
print(f'Logistic Regression 대비 성능: {"우수" if mlp_score > best_score else "낮음"}')

MLP (k=10) CV5 점수: 0.711
Logistic Regression 대비 성능: 낮음


## B-3. SVM 개념

SVM은 **두 그룹 사이에 제일 넉넉한 경계선**을 찾아. 경계에 가까운 점(support vector)과의 거리(margin)를 최대로.

핵심 2개만 기억하면 돼.

| 이름 | 역할 | 크면 | 작으면 |
|------|------|------|--------|
| `C` | regularization의 역수 | 학습 데이터에 바짝 맞춤 → overfitting↑ | 경계 부드러움 → underfitting 가능 |
| `kernel` | 경계 모양 | `linear`: 직선 / `rbf`: 종 모양 곡선 / `poly`: 다항식 | — |

- 📖 SVC: <https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html>
- 📖 SVM 해설: <https://scikit-learn.org/stable/modules/svm.html>


## ✏️ 실습 3 — 같은 K로 SVM (`C` × `kernel`)

`best_k`로 descriptor 뽑고, `C`와 `kernel` 조합을 표처럼 찍어 봐.

**힌트**
- `from sklearn.svm import SVC`
- 이중 for: `for C in [0.1, 1, 10]: for kernel in ['linear', 'rbf', 'poly']:`
- RBF/poly는 스케일에 민감 → **`StandardScaler` 필수**
- 💡 `poly`는 경고 뜰 수 있는데 정상. 거슬리면 `import warnings; warnings.filterwarnings('ignore')`.


In [9]:
# SVM모델을 사용하여 다양한 설정(C, Kernel)중 어떤 조합이 가장 높은 성능을 내는지 확인하는 단계
# 실습1에서 찾은 k_best를 그대로 활용하여 코드 작성
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import warnings

# TODO: best_k로 X_sel 만들고, C × kernel 조합 CV5 출력
#       예시 출력: 'C=0.1, kernel=linear : 0.775'

# poly 커널 연산 시 발생할 수 있는 경고 메시지를 무시합니다.
warnings.filterwarnings('ignore')

# 1. 실습 1에서 찾은 best_k로 특징 선택 및 표준화
# (best_k가 정의되어 있어야 합니다.)
selector = SelectKBest(f_classif, k=best_k)
X_sel = selector.fit_transform(X, y)
X_scaled = StandardScaler().fit_transform(X_sel)

print(f"--- SVM 성능 비교 (K={best_k}) ---")

# 2. C(규제 강도)와 kernel(경계 모양) 조합을 바꿔가며 테스트
# C 값이 클수록 모델이 복잡해지고, 작을수록 단순해집니다.
for C in [0.1, 1, 10]:
    for kernel in ['linear', 'rbf', 'poly']:
        # SVM 모델 정의
        svm = SVC(C=C, kernel=kernel)
        
        # 5-fold 교차 검증 수행
        scores = cross_val_score(svm, X_scaled, y, cv=5)
        avg_score = scores.mean()
        
        # 결과 출력 (예: 'C=0.1, kernel=linear : 0.775')
        print(f'C={C:3}, kernel={kernel:6} : {avg_score.round(3)}')

--- SVM 성능 비교 (K=10) ---
C=0.1, kernel=linear : 0.775
C=0.1, kernel=rbf    : 0.668
C=0.1, kernel=poly   : 0.668
C=  1, kernel=linear : 0.689
C=  1, kernel=rbf    : 0.746
C=  1, kernel=poly   : 0.693
C= 10, kernel=linear : 0.711
C= 10, kernel=rbf    : 0.768
C= 10, kernel=poly   : 0.614


In [16]:
# 결과보고서 작성시
# 알고리즘별 best결과 한 줄씩 작성
# 각 노드별 가장 잘 나온 데이터만 표로 만들어서 작성
# 세부적인 폭 안에서 또 다시 그래프를 그리기

---
## 스스로 확인

세 실습을 다 채워 실행했으면, 아래 질문에 머릿속으로 답해 봐.

- (Q1) 최적 K는 몇? 왜?
- (Q2) 같은 K에서 MLP가 로지스틱보다 좋게 나올까? 이유는?
- (Q3) SVM에서 제일 좋은 `(C, kernel)` 조합은?


In [13]:
A1. 최적 K는 K=10
    현재 화합물 샘플이 39개로 매우 적기 때문에, 너무 많은 특징을 쓰면 모델이 overfitting이 발생하여 교차 검증 점수가 떨어지게 됨.
    k=10일때 최고점이 출력됨.
A2. 데이터 부족 : MLP(신경망)는 복잡한 패턴을 배우기 위해 방대한 데이터가 필요하지만, 현재 데이터는 39개밖에 없다.
    모델의 복잡도 : 샘플이 적을 때는 MLP처럼 복잡한 모델보다 로지스틱 회귀처럼 단순한 모델이 더 안정적이고 정확한 예측을 수행하는 경우가 많음.
A3. c=0.1, kernel = 'linear' 조합이 0.775로 가장 좋음.

SyntaxError: invalid syntax (3173544580.py, line 1)